In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split

In [2]:
X_train = pd.read_csv('../data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('../data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('../data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('../submissions/sample_submission.csv',index_col='ROW_ID')

In [13]:
X_train.columns

Index(['TS', 'ALLOCATION', 'RET_20', 'RET_19', 'RET_18', 'RET_17', 'RET_16',
       'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10', 'RET_9',
       'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2', 'RET_1',
       'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18',
       'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15',
       'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12',
       'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9',
       'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6',
       'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3',
       'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'MEDIAN_DAILY_TURNOVER', 'GROUP'],
      dtype='str')

In [3]:
RET_features = [f'RET_{i}' for i in range(1,21)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,21)]
TURNOVER_features = ['MEDIAN_DAILY_TURNOVER']

In [4]:
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i]].mean(axis = 1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
    X_test[ f'AVERAGE_PERF_{i}'] = X_test[RET_features[:i]].mean(axis = 1)
    X_test[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_test.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')

for i in [20]:
    X_train[ f'STD_PERF_{i}'] = X_train[RET_features[:i]].std(axis = 1)
    X_train[ f'ALLOCATIONS_STD_PERF_{i}'] = X_train.groupby('TS')[ f'STD_PERF_{i}'].transform('mean')
    
    X_test[ f'STD_PERF_{i}'] = X_test[RET_features[:i]].std(axis = 1)
    X_test[ f'ALLOCATIONS_STD_PERF_{i}'] = X_test.groupby('TS')[ f'STD_PERF_{i}'].transform('mean')


In [5]:
for i in [3,5,10,20]:
    X_train[f'FLOW_MEAN_{i}'] = X_train[SIGNED_VOLUME_features[:i]].mean(axis=1)
    X_test[f'FLOW_MEAN_{i}'] = X_test[SIGNED_VOLUME_features[:i]].mean(axis=1)

for i in [5,10,20]:
    X_train[f'FLOW_IMBALANCE_{i}'] = X_train[SIGNED_VOLUME_features[:i]].sum(axis=1)
    X_test[f'FLOW_IMBALANCE_{i}'] = X_test[SIGNED_VOLUME_features[:i]].sum(axis=1)

for i in [10,20]:
    X_train[f'FLOW_STD_{i}'] = X_train[SIGNED_VOLUME_features[:i]].std(axis=1)
    X_test[f'FLOW_STD_{i}'] = X_test[SIGNED_VOLUME_features[:i]].std(axis=1)

X_train['FLOW_SHOCK'] = X_train['SIGNED_VOLUME_1'] - X_train[SIGNED_VOLUME_features[:5]].mean(axis=1)
X_test['FLOW_SHOCK'] = X_test['SIGNED_VOLUME_1'] - X_test[SIGNED_VOLUME_features[:5]].mean(axis=1)

In [6]:
features = RET_features + SIGNED_VOLUME_features + TURNOVER_features
features = features + [ f'AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'STD_PERF_{i}' for i in [20]]
features = features + [ f'ALLOCATIONS_STD_PERF_{i}' for i in [20]]

In [7]:
flow_features = [
    'FLOW_MEAN_3',
    'FLOW_MEAN_5',
    'FLOW_MEAN_10',
    'FLOW_MEAN_20',
    'FLOW_IMBALANCE_5',
    'FLOW_IMBALANCE_10',
    'FLOW_IMBALANCE_20',
    'FLOW_STD_10',
    'FLOW_STD_20',
    'FLOW_SHOCK'
]

features += flow_features

In [8]:
# 1. GROUP one-hot encoding
group_train = pd.get_dummies(X_train['GROUP'], prefix='GROUP')
group_test = pd.get_dummies(X_test['GROUP'], prefix='GROUP')

# 2. align train / test columns
group_train, group_test = group_train.align(group_test, join='outer', axis=1, fill_value=0)

# 3. concat
X_train = pd.concat([X_train, group_train], axis=1)
X_test = pd.concat([X_test, group_test], axis=1)

# 4. add to feature list
group_features = list(group_train.columns)
features = features + group_features

In [20]:
features

['RET_1',
 'RET_2',
 'RET_3',
 'RET_4',
 'RET_5',
 'RET_6',
 'RET_7',
 'RET_8',
 'RET_9',
 'RET_10',
 'RET_11',
 'RET_12',
 'RET_13',
 'RET_14',
 'RET_15',
 'RET_16',
 'RET_17',
 'RET_18',
 'RET_19',
 'RET_20',
 'SIGNED_VOLUME_1',
 'SIGNED_VOLUME_2',
 'SIGNED_VOLUME_3',
 'SIGNED_VOLUME_4',
 'SIGNED_VOLUME_5',
 'SIGNED_VOLUME_6',
 'SIGNED_VOLUME_7',
 'SIGNED_VOLUME_8',
 'SIGNED_VOLUME_9',
 'SIGNED_VOLUME_10',
 'SIGNED_VOLUME_11',
 'SIGNED_VOLUME_12',
 'SIGNED_VOLUME_13',
 'SIGNED_VOLUME_14',
 'SIGNED_VOLUME_15',
 'SIGNED_VOLUME_16',
 'SIGNED_VOLUME_17',
 'SIGNED_VOLUME_18',
 'SIGNED_VOLUME_19',
 'SIGNED_VOLUME_20',
 'MEDIAN_DAILY_TURNOVER',
 'AVERAGE_PERF_3',
 'AVERAGE_PERF_5',
 'AVERAGE_PERF_10',
 'AVERAGE_PERF_15',
 'AVERAGE_PERF_20',
 'ALLOCATIONS_AVERAGE_PERF_3',
 'ALLOCATIONS_AVERAGE_PERF_5',
 'ALLOCATIONS_AVERAGE_PERF_10',
 'ALLOCATIONS_AVERAGE_PERF_15',
 'ALLOCATIONS_AVERAGE_PERF_20',
 'STD_PERF_20',
 'ALLOCATIONS_STD_PERF_20',
 'FLOW_MEAN_3',
 'FLOW_MEAN_5',
 'FLOW_MEAN_10',
 'F

In [9]:
train_dates = np.sort(X_train["TS"].unique())

kf = KFold(n_splits=8, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(train_dates), 1):

    train_ts = train_dates[train_idx]
    valid_ts = train_dates[valid_idx]

    train_mask = X_train["TS"].isin(train_ts)
    valid_mask = X_train["TS"].isin(valid_ts)

    X_tr = X_train.loc[train_mask, features].fillna(0)
    X_va = X_train.loc[valid_mask, features].fillna(0)

    y_tr = (y_train.loc[train_mask].values.ravel() > 0).astype(int)
    y_va = (y_train.loc[valid_mask].values.ravel() > 0).astype(int)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
                max_iter=2000,
                C=1,
                # l1_ratio=1,
                # solver="liblinear"
        ))
    ])
    # model = LogisticRegression(max_iter=2000, C=1, l1_ratio=1, solver="saga")
    model.fit(X_tr, y_tr)

    pred = model.predict(X_va)

    acc = accuracy_score(y_va, pred)

    scores.append(acc)
    print(f"LogisticRegression Fold {fold}: {acc:.4f}")

mean_acc = np.mean(scores)
std_acc = np.std(scores)

print(f"Mean accuracy: {mean_acc:.4f}")
print(f"Std: {std_acc:.4f}")
print(f"Range: [{mean_acc-std_acc:.4f}, {mean_acc+std_acc:.4f}]")

LogisticRegression Fold 1: 0.5213
LogisticRegression Fold 2: 0.5223
LogisticRegression Fold 3: 0.5151
LogisticRegression Fold 4: 0.5242
LogisticRegression Fold 5: 0.5089
LogisticRegression Fold 6: 0.5213
LogisticRegression Fold 7: 0.5249
LogisticRegression Fold 8: 0.5255
Mean accuracy: 0.5204
Std: 0.0053
Range: [0.5151, 0.5257]


In [ ]:
# convert y to binary classification
y_train_bin = (y_train > 0).astype(int).values.ravel()

# define the model
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, C=1))
])

# train the model with full train set data
log_model.fit(X_train[features].fillna(0), y_train_bin)

# predict the probs
preds_prob = log_model.predict_proba(X_test[features].fillna(0))[:,1]

# convert the result into dataframe
preds_log = pd.DataFrame(
    preds_prob,
    index=sample_submission.index,
    columns=['target']
)

# threshold → submission
(preds_log > 0.5).astype(int).to_csv(
    '../submissions/preds_logistic_add_volFLOW_group.csv'
)

In [10]:
import xgboost as xgb


train_dates = np.sort(X_train["TS"].unique())

kf = KFold(n_splits=8, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(train_dates), 1):

    train_ts = train_dates[train_idx]
    valid_ts = train_dates[valid_idx]

    train_mask = X_train["TS"].isin(train_ts)
    valid_mask = X_train["TS"].isin(valid_ts)

    X_tr = X_train.loc[train_mask, features].fillna(0)
    X_va = X_train.loc[valid_mask, features].fillna(0)

    y_tr = (y_train.loc[train_mask].values.ravel() > 0).astype(int)
    y_va = (y_train.loc[valid_mask].values.ravel() > 0).astype(int)

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_tr, y_tr)

    pred = model.predict(X_va)

    acc = accuracy_score(y_va, pred)

    scores.append(acc)
    print(f"Fold {fold}: {acc:.4f}")

mean_acc = np.mean(scores)
std_acc = np.std(scores)

print(f"Mean accuracy: {mean_acc:.4f}")
print(f"Std: {std_acc:.4f}")
print(f"Range: [{mean_acc-std_acc:.4f}, {mean_acc+std_acc:.4f}]")

Fold 1: 0.5257
Fold 2: 0.5233
Fold 3: 0.5186
Fold 4: 0.5314
Fold 5: 0.5111
Fold 6: 0.5227
Fold 7: 0.5224
Fold 8: 0.5306
Mean accuracy: 0.5232
Std: 0.0061
Range: [0.5171, 0.5293]


In [11]:

# convert target to binary classification labels
y_train_bin = (y_train.values.ravel() > 0).astype(int)

# define the XGboost model (using parameters that performed well in CV)
xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42
)

# Train the model on the full training dataset
xgb_model.fit(
    X_train[features].fillna(0),
    y_train_bin
)

# predict probabilities for the test set
preds_prob = xgb_model.predict_proba(
    X_test[features].fillna(0)
)[:,1]

# convert predictions to DataFrame with correct index
preds_xgb = pd.DataFrame(
    preds_prob,
    index=sample_submission.index,
    columns=["target"]
)

# threshold -> binary prediction
(preds_xgb > 0.5).astype(int).to_csv(
    "../submissions/preds_xgb_add_volFLOW_group.csv"
)

In [13]:
from sklearn.linear_model import Ridge

train_dates = np.sort(X_train["TS"].unique())

kf = KFold(n_splits=8, shuffle = True, random_state = 42)

scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(train_dates), 1):

    train_ts = train_dates[train_idx]
    valid_ts = train_dates[valid_idx]

    train_mask = X_train["TS"].isin(train_ts)
    valid_mask = X_train["TS"].isin(valid_ts)

    X_tr = X_train.loc[train_mask, features].fillna(0)
    X_va = X_train.loc[valid_mask, features].fillna(0)

    y_tr = y_train.loc[train_mask].values.ravel()
    y_va = y_train.loc[valid_mask].values.ravel()

    model = Ridge(alpha = 1)
    model.fit(X_tr, y_tr)

    pred = model.predict(X_va)

    acc = accuracy_score(
        (y_va > 0).astype(int),
        (pred > 0).astype(int),
    )

    scores.append(acc)
    print(f"Fold {fold}: {acc:.4f}")

mean_acc = np.mean(scores)
std_acc = np.std(scores)

print(f"Mean accuracy: {mean_acc:.4f}")
print(f"Std: {std_acc:.4f}")
print(f"Range: [{mean_acc-std_acc:.4f}, {mean_acc+std_acc:.4f}]")

Fold 1: 0.5215
Fold 2: 0.5215
Fold 3: 0.5179
Fold 4: 0.5221
Fold 5: 0.5081
Fold 6: 0.5200
Fold 7: 0.5220
Fold 8: 0.5260
Mean accuracy: 0.5199
Std: 0.0049
Range: [0.5149, 0.5248]


In [15]:
new_ridge = Ridge(alpha=1)

new_ridge.fit(X_train[features].to_numpy(na_value=0),y_train.to_numpy(na_value=0))

preds_ridge = pd.DataFrame(new_ridge.predict(X_test[features].fillna(0).to_numpy(na_value=0)), index = sample_submission.index,columns=['target'])


In [16]:
blend = 0.5 * preds_ridge + 0.5 * preds_xgb

(blend > 0.5).astype(int).to_csv(
    "../submissions/preds_ridge_xgb_blend.csv"
)

In [18]:
preds_xgb

,target
ROW_ID,
527073,0.485735
527074,0.509362
527075,0.509646
527076,0.477304
527077,0.467359
...,...
558938,0.491298
558939,0.480106
558940,0.487334


In [26]:
ridge_raw = preds_ridge["target"].values
ridge_prob = 1 / (1 + np.exp(-ridge_raw))

xgb_prob = preds_xgb["target"].values

blend_score = 0.5 * ridge_prob + 0.5 * xgb_prob
# submission = (blend_prob > 0.5).astype(int)

In [27]:
submission = pd.DataFrame(
    (blend_score > 0.5).astype(int),
    index=sample_submission.index,
    columns=["target"]
)

submission.to_csv("../submissions/preds_ridge_xgb_blend.csv")

In [30]:
coef_df = pd.DataFrame({
    "feature": features,
    "coef": new_ridge.coef_
}).sort_values("coef", key=abs, ascending=False)


coef_df.to_csv("ridge_coefficients.csv", index=False)

In [33]:
print(coef_df.to_string())

                        feature      coef
0                         RET_1  0.066462
3                         RET_4 -0.020617
1                         RET_2 -0.016949
6                         RET_7  0.014641
8                         RET_9  0.014328
7                         RET_8  0.013574
2                         RET_3 -0.013266
41               AVERAGE_PERF_3  0.012083
46   ALLOCATIONS_AVERAGE_PERF_3  0.011027
17                       RET_18 -0.008857
5                         RET_6  0.007889
15                       RET_16  0.007669
43              AVERAGE_PERF_10  0.006203
48  ALLOCATIONS_AVERAGE_PERF_10  0.004802
44              AVERAGE_PERF_15  0.004751
13                       RET_14  0.004444
52      ALLOCATIONS_STD_PERF_20  0.003988
47   ALLOCATIONS_AVERAGE_PERF_5  0.003980
12                       RET_13  0.003979
18                       RET_19  0.003843
16                       RET_17 -0.003767
49  ALLOCATIONS_AVERAGE_PERF_15  0.003557
10                       RET_11  0